# धडा 12 - एजंट स्क्रॅचपॅडसह चॅट प्रवर्ग कमी करणे

हा नोटबुक मायक्रोसॉफ्ट एजंट फ्रेमवर्क वापरून लांबच्या संभाषणांमधील संदर्भ कसे व्यवस्थापित करायचे हे दाखवतो. संभाषणे वाढत असताना, टोकन संख्या वाढते — अखेरीस मॉडेलच्या संदर्भ विंडोपेक्षा जास्त होते. आम्ही हे **संदर्भ सारांश नमुना** आणि **एजंट स्क्रॅचपॅड** वापरून पर्सिस्टंट स्मृतीसाठी हाताळतो.

## तुम्हाला काय शिकायला मिळेल:
1. **संदर्भ व्यवस्थापन का महत्त्वाचे आहे**: टोकन मर्यादा आणि संदर्भ विंडो समजून घेणे
2. **संदर्भ-अनुभवी एजंट्स**: स्वतःच्या संभाषण संदर्भाचे व्यवस्थापन करणारे एजंट तयार करणे
3. **संदर्भ सारांश नमुना**: संभाषण इतिहास संक्षिप्त करण्यासाठी साधने वापरणे
4. **एजंट स्क्रॅचपॅड**: संदर्भ कमी होण्यावर टिकून राहणारी स्मृती

## पूर्वअट:
- एझ्युअर ओपनएआय सेटअप वातावरण चलांसह संरक्षित
- मागील धड्यांमधील प्राथमिक एजंट संकल्पनांचा समज


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## संदर्भ व्यवस्थापन का महत्त्वाचे आहे

प्रत्येक LLM कडे एक मर्यादित **संदर्भ विंडो** असते — एका विनंतीत प्रक्रिया करू शकणारा कमाल टोकन संख्या. एक बहु-चरण संभाषण चालू असताना:

- प्रत्येक वापरकर्ता संदेश आणि सहाय्यक प्रतिसादासह **टोकनांची संख्या रेषीयरीत्या वाढते**.
- **प्रॉम्प्ट टोकन खर्चावर नियंत्रण ठेवतात** कारण संपूर्ण इतिहास प्रत्येक टप्प्यावर परत पाठविला जातो.
- शेवटी संभाषण **संदर्भ विंडो ओलांडते** आणि मॉडेल कापणी करते किंवा त्रुटी दर्शवते.

### संदर्भ व्यवस्थापनासाठी धोरणे

| धोरण | ते कसे कार्य करते | तोटा |
|---|---|---|
| **कापणी (Truncation)** | सर्वात जुने संदेश कापून टाकणे | सुरुवातीचा संदर्भ गमावलेला |
| **सारांश (Summarization)** | जुने संदेश संक्षिप्त सारांशात रूपांतरित करणे | काही तपशील हरवतात, पण मुख्य मुद्दे ठेवले जातात |
| **स्क्रॅचपॅड / बाह्य स्मरणशक्ती** | संभाषणाबाहेर की माहिती साठवणे | टूल कॉल्स आवश्यक, पण कोणत्याही कपातीला तोंड देतो |

या नोटबुकमध्ये आम्ही **सारांश** व **स्क्रॅचपॅड टूल** एकत्र वापरतो जेणेकरून एजंटला संभाषणाचा प्रवाह कायम ठेवता येईल जरी संभाषण इतिहास संक्षिप्त केला गेला असला तरी.


## संदर्भ-जाणणारा एजंट तयार करणे


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## दीर्घ संभाषणाचा अनुकरण करणे

संदर्भ कसा जमा होतो हे पाहण्यासाठी आपण एक मल्टी-टर्न संभाषण पाहूया. एजंटने टर्न्स दरम्यान महत्त्वाची माहिती (आवडी, बजेट, प्रवासाच्या तारखा) जपली पाहिजे आणि सातत्य दाखवले पाहिजे.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

एजंट कसा पूर्वीच्या संवादांमधील संदर्भ जपतो ते लक्षात घ्या — त्याला जपान, सुशी, मंदिरे, छायाचित्रण, $3000 बजेट, एकट्याने प्रवास आणि एप्रिलचा कालावधी माहिती आहे. एका लहान संभाषणात हे चांगले काम करते, पण संभाषण वाढल्यावर पूर्ण इतिहास पुन्हा पाठवणे महागडं ठरते.

संदर्भ साठवण्याचा परिणाम पाहण्यासाठी आपण अधिक संवाद घेऊन पुढे जाऊया:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## संदर्भ संक्षेपण नमुना

संभाषण वाढत गेल्यामुळे, आम्ही एक **संक्षेपण साधन** वापरू शकतो जे जमा केलेल्या संदर्भाला संक्षिप्त स्वरूपात रूपांतरित करते. एजंट हे साधन कॉल करतो जेणेकरुन महत्त्वाच्या प्राधान्य गोष्टी नोंदवल्या जातात, त्यामुळे अगदी जुनी संदेशे काढून टाकली तरीही आवश्यक माहिती जतन होते.

हा नमुना अधिक सुस्पष्ट इतिहास कमी करण्याचा पाया आहे:
1. एजंट संभाषणातून मुख्य तथ्ये ओळखतो
2. तो संक्षेपण साधन कॉल करतो जेणेकरुन ती टिकवून ठेवता येतील
3. जुनी संदेशे सुरक्षितपणे काढून टाकता येतात कारण संक्षेप महत्त्वाच्या गोष्टी पकडते

खाली आम्ही `summarize_preferences` साधन परिभाषित करतो ज्याला एजंट कॉल करून त्याने जे शिकलं आहे त्याचं एक संक्षिप्त सारांश नोंदवू शकतो.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## सारांश

या धड्यात तुम्ही Microsoft Agent Framework वापरून दीर्घकाळ चालणाऱ्या एजंट संभाषणांमध्ये संदर्भ कसे व्यवस्थापित करायचे हे शिकलात:

### मुख्य संकल्पना
- **संदर्भ विंडोज मर्यादित असतात** — संभाषण इतिहासातील प्रत्येक टोकनचे शुल्क आकारले जाते आणि हे मर्यादेतील गणनेत येते.
- **सारांश साधने** एजंटला जमा केलेल्या संदर्भाचा संक्षिप्त सारांश तयार करण्यास मदत करतात, ज्यामुळे टोकन वापर कमी होतो आणि आवश्यक माहिती जपली जाते.
- **एजंट स्क्रॅचपॅड्स** दीर्घकाल टिकणारी बाह्य स्मृती देतात, जी संभाषण घटनेनंतरही टिकून राहते.

### तुम्ही काय तयार केले
- एक **संदर्भ-जागरूक एजंट** जो बहु-टर्न संभाषणांमध्ये सलगता राखतो
- एक **सारांश साधन** (`summarize_preferences`) जे महत्त्वाची वापरकर्ता माहिती संक्षिप्त स्वरूपात नोंदवते
- एक **बहु-टर्न संभाषण** ज्यामध्ये संदर्भ टिकविणे आणि बदल हाताळणे दर्शवले आहे

### वास्तविक जगातील अनुप्रयोग
- **ग्राहक सेवा बॉट्स**: दीर्घकालीन समर्थन सत्रांमध्ये पसंती लक्षात ठेवतात
- **वैयक्तिक सहाय्यक**: चालू प्रकल्पांचा मागोवा घेतात, संदर्भ पुन्हा समजावून सांगण्याची गरज नाही
- **शैक्षणिक शिक्षक**: अनेक संवादांमध्ये विद्यार्थ्यांची प्रगती राखतात

### पुढील पावले
- फाइल-आधारित टिकावदार कार्यक्षमतेसह पूर्ण स्क्रॅचपॅड साधन अंमलात आणा
- सारांशानंतर आपोआप इतिहास संक्षेपित करण्याची प्रणाली जोडा
- सेमांटिक मेमरी शोधासाठी व्हेक्टर डेटाबेससह संयोजन करा
- एजंट तयार करा जे संपूर्ण संदर्भासह दिवसांनंतर संभाषण पुन्हा सुरू करू शकतात


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
